In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

In [ ]:
data = pd.read_csv("Advertising_Original.csv")
data.head()

In [ ]:
data.shape, data.info(), data.isnull().sum(), data.duplicated().sum()

In [ ]:
data = data.drop(columns=["Unnamed: 0"], errors="ignore").drop_duplicates().copy()
data.to_csv("Advertising_Cleaned.csv", index=False)
data.describe()

In [ ]:
features = ["TV", "Radio", "Newspaper"]
X, y = data[features], data["Sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(data["Sales"], bins=18, edgecolor="black")
plt.xlabel("Sales"); plt.ylabel("Frequency"); plt.title("Distribution of Sales")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,4))
for ax, feature in zip(axes, features):
    ax.scatter(data[feature], data["Sales"], alpha=0.75)
    ax.set_xlabel(feature + " Advertising"); ax.set_ylabel("Sales"); ax.set_title(feature + " vs Sales")
plt.tight_layout(); plt.show()

In [ ]:
models = {"Linear Regression": LinearRegression(), "Random Forest Regressor": RandomForestRegressor(n_estimators=400, random_state=42)}
results, fitted, predictions = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name], predictions[name] = model, pred
    results.append({"Model": name, "MAE": mean_absolute_error(y_test,pred), "RMSE": mean_squared_error(y_test,pred)**0.5, "R2_Score": r2_score(y_test,pred)})
metrics = pd.DataFrame(results).sort_values("R2_Score", ascending=False)
metrics

In [ ]:
best_name = metrics.iloc[0]["Model"]
best_model, best_pred = fitted[best_name], predictions[best_name]
best_name

In [ ]:
pd.DataFrame({"Actual_Sales": y_test.values, "Predicted_Sales": best_pred}).to_csv("Test_Set_Predictions.csv", index=False)
metrics.to_csv("Model_Evaluation_Metrics.csv", index=False)

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test,best_pred,alpha=0.8)
lim=[min(y_test.min(),best_pred.min()),max(y_test.max(),best_pred.max())]
plt.plot(lim,lim,linestyle="--")
plt.xlabel("Actual Sales"); plt.ylabel("Predicted Sales"); plt.title("Actual vs Predicted Sales")
plt.tight_layout(); plt.show()

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=30, random_state=42, scoring="r2")
feature_importance = pd.DataFrame({"Feature": features, "Importance_Mean": perm.importances_mean, "Importance_Std": perm.importances_std}).sort_values("Importance_Mean", ascending=False)
feature_importance.to_csv("Feature_Importance.csv", index=False)
feature_importance